<a href="https://colab.research.google.com/github/joaoestella/IA-Course-Projects/blob/main/assignments/CP2_Otimizacao_de_Hiperparametros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CP 2

### 🇧🇷 Português
Utilize Grid Search ou Random Search para resolver o problema de detecção de fraudes, utilizando o dataset: https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv

Treine um modelo com os melhores parâmetros encontrados e mostre os erros de treinamento e teste. Faça uma discussão sobre a performance do modelo de acordo com a sua análise dos valores de erro obtidos.

---

### 🇺🇸 English
Use Grid Search or Random Search to solve the fraud detection problem using the dataset: https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv

Train a model with the best parameters found and show the training and test errors. Provide a discussion about the model’s performance based on your analysis of the obtained error values.

In [ ]:
# Bibliotecas principais
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Pré-processamento e Modelos
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [ ]:
df = pd.read_csv("https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv")

In [ ]:
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


In [ ]:
# Visualizar o dataset
print(df.head())
print(df['Class'].value_counts())  # Verificar desbalanceamento


   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

In [ ]:
# Separar features e target
X = df.drop('Class', axis=1)
y = df['Class']

# Dividir treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Tamanho treino: {X_train.shape}, Tamanho teste: {X_test.shape}")

Tamanho treino: (227845, 30), Tamanho teste: (56962, 30)


In [ ]:
# Criar modelo base de Decision Tree
dt = DecisionTreeClassifier(random_state=42)

In [ ]:
# Definindo a grade de parâmetros para Grid Search
param_grid = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

# Aplicar Grid Search
grid_search = GridSearchCV(dt, param_grid, scoring='accuracy', cv=3, n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Melhores parâmetros do Grid Search:", grid_search.best_params_)


Melhores parâmetros do Grid Search: {'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 10}


In [ ]:
# Definindo a distribuição de parâmetros para Random Search
from scipy.stats import randint

param_dist = {
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 20),
}

# Aplicar Randomized Search
random_search = RandomizedSearchCV(dt, param_distributions=param_dist, n_iter=10,
                                   scoring='accuracy', cv=3, random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

print("Melhores parâmetros do Random Search:", random_search.best_params_)


Melhores parâmetros do Random Search: {'max_depth': 13, 'min_samples_leaf': 4, 'min_samples_split': 9}


In [ ]:
# Escolher o melhor entre Grid Search e Random Search
best_model = grid_search.best_estimator_  # ou random_search.best_estimator_

# Treinar novamente no conjunto de treino
best_model.fit(X_train, y_train)


DecisionTreeClassifier(criterion='entropy', max_depth=5, min_samples_leaf=2,
                       min_samples_split=10, random_state=42)

In [ ]:
# Avaliar
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)

# Métricas
acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

print(f"Accuracy Treino: {acc_train:.4f}")
print(f"Accuracy Teste: {acc_test:.4f}")

# Matriz de Confusão
print("\nMatriz de Confusão (Teste):")
print(confusion_matrix(y_test, y_pred_test))

# Relatório de Classificação
print("\nRelatório de Classificação (Teste):")
print(classification_report(y_test, y_pred_test))


Accuracy Treino: 0.9996
Accuracy Teste: 0.9995

Matriz de Confusão (Teste):
[[56856     8]
 [   21    77]]

Relatório de Classificação (Teste):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.91      0.79      0.84        98

    accuracy                           1.00     56962
   macro avg       0.95      0.89      0.92     56962
weighted avg       1.00      1.00      1.00     56962



# Análise do Modelo

O modelo treinado utilizando Árvore de Decisão apresentou uma performance muito boa. A acurácia no conjunto de treino foi de 99,96% e no conjunto de teste foi de 99,95%, indicando que o modelo acertou a grande maioria das previsões.

Mesmo sendo um problema de detecção de fraude, que possui bem menos exemplos positivos, o modelo conseguiu identificar corretamente 77 fraudes e errou 21, o que é um bom resultado considerando o desbalanceamento do dataset.

A precisão para a classe 1, fraude, foi de 91%, enquanto o recall foi de 79%. Isso significa que, quando o modelo prevê uma fraude, na maioria das vezes ele está correto, e também consegue identificar uma boa parte das fraudes existentes. O f1-score foi de 84%, indicando um bom equilíbrio entre precisão e recall.

---

The model trained using a Decision Tree showed very good performance. The training accuracy was 99.96% and the test accuracy was 99.95%, indicating that the model correctly predicted most cases.

Even though this is a fraud detection problem, which has far fewer positive examples, the model correctly identified 77 fraud cases and missed 21, which is a good result considering the dataset is imbalanced.

The precision for class 1, fraud, was 91%, while the recall was 79%. This means that when the model predicts fraud, it is correct most of the time, and it is also able to detect a good portion of the fraud cases. The f1-score was 84%, showing a good balance between precision and recall.